In [12]:
# =============================================================================
# STW7085CEM — Advanced Machine Learning
# Project: Predicting Epilepsy Diagnosis using Structured EHR Data
# Methods: Gaussian Process Classification vs Decision Tree
# Dataset: NHANES Structured EHR Data
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')

### configuration & Paths

In [13]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# os.getcwd() returns the notebook directory — __file__ is not available in Jupyter
BASE_DIR    = os.getcwd()
DATA_PATH   = os.path.join(BASE_DIR, 'results', 'data_epilepsy.csv')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

SELECTED_FEATURES = ['MCQ080', 'MCQ160B', 'DPQ010', 'PHQ9_TOTAL', 'PAQ605', 'LBXSGL']
TARGET            = 'EPILEPSY'
BINARY_COLS       = ['MCQ080', 'MCQ160B']   # NHANES: 1=Yes, 2=No → recode to 1/0

plt.rcParams.update({
    'figure.dpi':      150,
    'axes.titlesize':  13,
    'axes.labelsize':  11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'font.family':     'sans-serif',
})

### LOAD DATASET

In [14]:
print("=" * 60)
print("STEP 1 — LOADING DATASET")
print("=" * 60)
 
df = pd.read_csv(DATA_PATH)
 
print(f"\nRaw dataset shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())

STEP 1 — LOADING DATASET

Raw dataset shape: (29400, 11)

First 5 rows:
    SEQN  MCQ080  MCQ160B  DPQ010  PHQ9_TOTAL  SLD010H  PAQ605  LBXSGL  \
0  73557     1.0      2.0     1.0         1.0      7.0     2.0   554.0   
1  73558     2.0      2.0     2.0         2.0      9.0     2.0   219.0   
2  73559     2.0      2.0     0.0         0.0      8.0     2.0   183.0   
3  73560     NaN      NaN     NaN         NaN      NaN     NaN     NaN   
4  73561     2.0      2.0     2.0         9.0      9.0     2.0   104.0   

   LBXGLU  EPILEPSY CYCLE  
0     NaN         0     H  
1     NaN         0     H  
2   193.0         0     H  
3     NaN         0     H  
4   107.0         0     H  

Data types:
SEQN            int64
MCQ080        float64
MCQ160B       float64
DPQ010        float64
PHQ9_TOTAL    float64
SLD010H       float64
PAQ605        float64
LBXSGL        float64
LBXGLU        float64
EPILEPSY        int64
CYCLE             str
dtype: object

Missing values per column:
SEQN              

### FEATURE SELECTION

In [15]:
df = df[SELECTED_FEATURES + [TARGET]]

print(f"\nSelected {len(SELECTED_FEATURES)} input features + 1 target.")
print(f"Dataset shape after feature selection: {df.shape}")


Selected 6 input features + 1 target.
Dataset shape after feature selection: (29400, 7)


### DATA PREPROCESSING

In [16]:
print("\n" + "=" * 60)
print("STEP 2 — PREPROCESSING")
print("=" * 60)

# Drop rows with any missing value in the selected columns
before = len(df)
df = df.dropna()
print(f"\nRows dropped (missing values)      : {before - len(df)}")
print(f"Rows remaining                     : {len(df)}")

# Recode NHANES binary columns: 1=Yes → 1, 2=No → 0
# Values 7 (refused) or 9 (don't know) become NaN and are dropped
for col in BINARY_COLS:
    df[col] = df[col].replace({1: 1, 2: 0})
    df[col] = pd.to_numeric(df[col], errors='coerce')

before = len(df)
df = df.dropna()
print(f"Rows dropped (invalid binary codes): {before - len(df)}")
print(f"Final clean dataset shape          : {df.shape}")

# Class balance
class_counts = df[TARGET].value_counts().sort_index()
print(f"\nClass distribution:\n{class_counts}")
print(f"Epilepsy prevalence: {class_counts[1] / len(df) * 100:.2f}%")

print("\nDescriptive statistics:")
print(df.describe().round(3))

# Save cleaned dataset for reference / citation in report
clean_path = os.path.join(RESULTS_DIR, 'epilepsy_cleaned.csv')
df.to_csv(clean_path, index=False)
print(f"\nClean dataset saved: epilepsy_cleaned.csv  ({df.shape[0]} rows)")


STEP 2 — PREPROCESSING

Rows dropped (missing values)      : 15283
Rows remaining                     : 14117
Rows dropped (invalid binary codes): 0
Final clean dataset shape          : (14117, 7)

Class distribution:
EPILEPSY
0    13731
1      386
Name: count, dtype: int64
Epilepsy prevalence: 2.73%

Descriptive statistics:
          MCQ080    MCQ160B     DPQ010  PHQ9_TOTAL     PAQ605     LBXSGL  \
count  14117.000  14117.000  14117.000   14117.000  14117.000  14117.000   
mean       0.386      0.049      0.415       3.356      1.786    104.488   
std        0.522      0.405      0.848       4.525      0.434     40.808   
min        0.000      0.000      0.000       0.000      1.000     19.000   
25%        0.000      0.000      0.000       0.000      2.000     87.000   
50%        0.000      0.000      0.000       2.000      2.000     94.000   
75%        1.000      0.000      1.000       5.000      2.000    105.000   
max        9.000      9.000      9.000      72.000      9.000   

### EXPLORATORY DATA ANALYSIS (EDA) PLOTS

In [17]:
print("\n" + "=" * 60)
print("STEP 3 — EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# --- Missing data bar chart (computed from raw CSV before preprocessing) ---
raw_df   = pd.read_csv(DATA_PATH)
miss_pct = raw_df[SELECTED_FEATURES + [TARGET]].isnull().mean() * 100

fig, ax = plt.subplots(figsize=(7, 4))
miss_pct[miss_pct > 0].sort_values().plot(kind='barh', ax=ax,
                                           color='#DD8452', edgecolor='white')
ax.axvline(5, color='red', linestyle='--', lw=1, label='5% threshold')
ax.set_title('Missing Data (%) by Feature — Raw Dataset')
ax.set_xlabel('Missing (%)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'missing_data.png'))
plt.close()
print("  Saved: missing_data.png")

# --- Class distribution bar chart ---
fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind='bar', color=['#4C72B0', '#DD8452'], ax=ax, edgecolor='white')
ax.set_title('Epilepsy Diagnosis — Class Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_xticklabels(['No Epilepsy (0)', 'Epilepsy (1)'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'class_distribution.png'))
plt.close()
print("  Saved: class_distribution.png")

# --- Feature correlation heatmap ---
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix (clean data)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'correlation_heatmap.png'))
plt.close()
print("  Saved: correlation_heatmap.png")

# --- Per-feature histograms split by class ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flatten(), SELECTED_FEATURES):
    for label, color in zip([0, 1], ['#4C72B0', '#DD8452']):
        ax.hist(df[df[TARGET] == label][feat], bins=20, alpha=0.6,
                color=color, label=f'Class {label}', edgecolor='white')
    ax.set_title(feat)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend()
plt.suptitle('Feature Distributions by Epilepsy Class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_distributions.png'), bbox_inches='tight')
plt.close()
print("  Saved: feature_distributions.png")

# --- Pairplot (500-row sample to keep it readable) ---
sample = df.sample(min(500, len(df)), random_state=RANDOM_STATE)
pplot  = sns.pairplot(sample, hue=TARGET, vars=SELECTED_FEATURES,
                      palette={0: '#4C72B0', 1: '#DD8452'},
                      plot_kws={'alpha': 0.4, 's': 20},
                      diag_kind='hist')
pplot.fig.suptitle('Pairplot — Selected Features by Epilepsy Class', y=1.01)
pplot.savefig(os.path.join(RESULTS_DIR, 'pairplot.png'), bbox_inches='tight')
plt.close('all')
print("  Saved: pairplot.png")


STEP 3 — EXPLORATORY DATA ANALYSIS
  Saved: missing_data.png
  Saved: class_distribution.png
  Saved: correlation_heatmap.png
  Saved: feature_distributions.png
  Saved: pairplot.png


### TRAIN / TEST SPLIT & SCALING

In [18]:
print("\n" + "=" * 60)
print("STEP 4 — TRAIN/TEST SPLIT AND FEATURE SCALING")
print("=" * 60)
 
X = df[SELECTED_FEATURES].values
y = df[TARGET].values
 
# Stratified split preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)
 
# StandardScaler: zero mean, unit variance — critical for GPC with RBF kernel
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)   # Fit ONLY on training data
X_test  = scaler.transform(X_test)        # Apply same transform to test data
 
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Features:         {X_train.shape[1]}")


STEP 4 — TRAIN/TEST SPLIT AND FEATURE SCALING
Training samples: 11293
Test samples:     2824
Features:         6


### MODEL 1: GAUSSIAN PROCESS CLASSIFICATION

In [19]:
print("\n" + "=" * 60)
print("MODEL 1 — GAUSSIAN PROCESS CLASSIFICATION (GPC)")
print("=" * 60)

# Kernel: Constant amplitude × Radial Basis Function (squared exponential)
# RBF captures smooth, nonlinear relationships in the feature space.
# The hyperparameters (amplitude, length_scale) are optimised via
# log-marginal-likelihood during fit().
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))

gpc_model = GaussianProcessClassifier(
    kernel=kernel,
    n_restarts_optimizer=5,     # Multiple restarts to find global optimum
    random_state=RANDOM_STATE,
    max_iter_predict=100
)

# GPC complexity is O(n³) — training on 11k rows would take several hours.
# Use all positive-class rows + an equal number of negatives → balanced ~616-row
# subsample. Evaluation still runs on the full 2824-row test set.
rng     = np.random.default_rng(RANDOM_STATE)
pos_idx = np.where(y_train == 1)[0]
neg_idx = np.where(y_train == 0)[0]
sub_idx = np.concatenate([
    pos_idx,
    rng.choice(neg_idx, size=len(pos_idx), replace=False),
])
X_train_gpc = X_train[sub_idx]
y_train_gpc = y_train[sub_idx]
print(f"\nGPC subsample: {len(sub_idx)} balanced rows "
      f"({len(pos_idx)} positive + {len(pos_idx)} negative)")

print("Fitting Gaussian Process Classifier ... (should finish in 30–90 s)")
gpc_model.fit(X_train_gpc, y_train_gpc)
print(f"Optimised kernel: {gpc_model.kernel_}")

# Predictions on full test set
gpc_pred = gpc_model.predict(X_test)
gpc_prob = gpc_model.predict_proba(X_test)[:, 1]   # Probability of class 1

# Metrics
gpc_acc  = accuracy_score(y_test, gpc_pred)
gpc_auc  = roc_auc_score(y_test, gpc_prob)
gpc_prec = precision_score(y_test, gpc_pred, zero_division=0)
gpc_rec  = recall_score(y_test, gpc_pred, zero_division=0)
gpc_f1   = f1_score(y_test, gpc_pred, zero_division=0)
gpc_cm   = confusion_matrix(y_test, gpc_pred)

print(f"\nAccuracy  : {gpc_acc:.4f}")
print(f"ROC-AUC   : {gpc_auc:.4f}")
print(f"Precision : {gpc_prec:.4f}")
print(f"Recall    : {gpc_rec:.4f}")
print(f"F1-Score  : {gpc_f1:.4f}")
print("\nConfusion Matrix:")
print(gpc_cm)
print("\nClassification Report:")
print(classification_report(y_test, gpc_pred, target_names=['No Epilepsy', 'Epilepsy']))

# 5-fold cross-validation (on subsample — full-set CV would take hours)
gpc_cv_scores = cross_val_score(
    GaussianProcessClassifier(kernel=kernel, random_state=RANDOM_STATE),
    X_train_gpc, y_train_gpc, cv=5, scoring='roc_auc'
)
print(f"\n5-Fold CV ROC-AUC (subsample): {gpc_cv_scores.mean():.4f} ± {gpc_cv_scores.std():.4f}")


MODEL 1 — GAUSSIAN PROCESS CLASSIFICATION (GPC)

GPC subsample: 618 balanced rows (309 positive + 309 negative)
Fitting Gaussian Process Classifier ... (should finish in 30–90 s)
Optimised kernel: 0.609**2 * RBF(length_scale=4.55)

Accuracy  : 0.6746
ROC-AUC   : 0.5497
Precision : 0.0312
Recall    : 0.3636
F1-Score  : 0.0574

Confusion Matrix:
[[1877  870]
 [  49   28]]

Classification Report:
              precision    recall  f1-score   support

 No Epilepsy       0.97      0.68      0.80      2747
    Epilepsy       0.03      0.36      0.06        77

    accuracy                           0.67      2824
   macro avg       0.50      0.52      0.43      2824
weighted avg       0.95      0.67      0.78      2824


5-Fold CV ROC-AUC (subsample): 0.5400 ± 0.0389


### MODEL 2: DECISION TREE CLASSIFIER

In [20]:
print("\n" + "=" * 60)
print("MODEL 2 — DECISION TREE CLASSIFIER (DTC)")
print("=" * 60)

dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    criterion='gini',
    class_weight='balanced',    # weights minority class ~35x to correct zero recall/F1
    random_state=RANDOM_STATE
)

print("\nFitting Decision Tree on full training set ...")
dt_model.fit(X_train, y_train)

# Predictions
dt_pred = dt_model.predict(X_test)
dt_prob = dt_model.predict_proba(X_test)[:, 1]

# Metrics
dt_acc  = accuracy_score(y_test, dt_pred)
dt_auc  = roc_auc_score(y_test, dt_prob)
dt_prec = precision_score(y_test, dt_pred, zero_division=0)
dt_rec  = recall_score(y_test, dt_pred, zero_division=0)
dt_f1   = f1_score(y_test, dt_pred, zero_division=0)
dt_cm   = confusion_matrix(y_test, dt_pred)

print(f"\nAccuracy  : {dt_acc:.4f}")
print(f"ROC-AUC   : {dt_auc:.4f}")
print(f"Precision : {dt_prec:.4f}")
print(f"Recall    : {dt_rec:.4f}")
print(f"F1-Score  : {dt_f1:.4f}")
print("\nConfusion Matrix:")
print(dt_cm)
print("\nClassification Report:")
print(classification_report(y_test, dt_pred, target_names=['No Epilepsy', 'Epilepsy']))

# Feature importances (Gini-based)
feat_imp = pd.Series(dt_model.feature_importances_, index=SELECTED_FEATURES)
print("\nDecision Tree Feature Importances (Gini):")
print(feat_imp.sort_values(ascending=False).round(4))

# 5-fold CV — also balanced so CV score reflects minority-class awareness
dt_cv_scores = cross_val_score(
    DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=RANDOM_STATE),
    X_train, y_train, cv=5, scoring='roc_auc'
)
print(f"\n5-Fold CV ROC-AUC: {dt_cv_scores.mean():.4f} ± {dt_cv_scores.std():.4f}")


MODEL 2 — DECISION TREE CLASSIFIER (DTC)

Fitting Decision Tree on full training set ...

Accuracy  : 0.6360
ROC-AUC   : 0.5769
Precision : 0.0352
Recall    : 0.4675
F1-Score  : 0.0655

Confusion Matrix:
[[1760  987]
 [  41   36]]

Classification Report:
              precision    recall  f1-score   support

 No Epilepsy       0.98      0.64      0.77      2747
    Epilepsy       0.04      0.47      0.07        77

    accuracy                           0.64      2824
   macro avg       0.51      0.55      0.42      2824
weighted avg       0.95      0.64      0.75      2824


Decision Tree Feature Importances (Gini):
LBXSGL        0.4843
PHQ9_TOTAL    0.3607
MCQ080        0.0660
MCQ160B       0.0339
PAQ605        0.0300
DPQ010        0.0252
dtype: float64

5-Fold CV ROC-AUC: 0.5290 ± 0.0450


In [21]:
print("\n" + "=" * 60)
print("STEP 5 — GENERATING VISUALISATIONS")
print("=" * 60)

# --- Confusion matrices (side-by-side) ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, cm, title, cmap in zip(
    axes,
    [gpc_cm, dt_cm],
    ['Gaussian Process Classifier', 'Decision Tree Classifier'],
    ['Blues', 'Oranges']
):
    ConfusionMatrixDisplay(confusion_matrix=cm,
                           display_labels=['No Epilepsy', 'Epilepsy']).plot(
        ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(f'Confusion Matrix\n{title}')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrices.png'))
plt.close()
print("  Saved: confusion_matrices.png")

# --- ROC Curves (both models) ---
fpr_gpc, tpr_gpc, _ = roc_curve(y_test, gpc_prob)
fpr_dt,  tpr_dt,  _ = roc_curve(y_test, dt_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_gpc, tpr_gpc, color='#4C72B0', lw=2, label=f'GPC  (AUC={gpc_auc:.3f})')
ax.plot(fpr_dt,  tpr_dt,  color='#DD8452', lw=2, linestyle='--',
        label=f'DTC  (AUC={dt_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.fill_between(fpr_gpc, tpr_gpc, alpha=0.1, color='#4C72B0')
ax.fill_between(fpr_dt,  tpr_dt,  alpha=0.1, color='#DD8452')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — GPC vs Decision Tree')
ax.legend(loc='lower right')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'roc_curves.png'))
plt.close()
print("  Saved: roc_curves.png")

# --- Metrics comparison bar chart ---
metrics_df = pd.DataFrame({
    'GPC': [gpc_acc, gpc_auc, gpc_prec, gpc_rec, gpc_f1],
    'DTC': [dt_acc,  dt_auc,  dt_prec,  dt_rec,  dt_f1]
}, index=['Accuracy', 'ROC-AUC', 'Precision', 'Recall', 'F1-Score'])
x, w = np.arange(len(metrics_df)), 0.35
fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - w/2, metrics_df['GPC'], w, label='GPC', color='#4C72B0', edgecolor='white')
bars2 = ax.bar(x + w/2, metrics_df['DTC'], w, label='DTC', color='#DD8452', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(metrics_df.index)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Performance Comparison: GPC vs Decision Tree')
ax.legend(); ax.grid(axis='y', linestyle='--', alpha=0.4)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'metrics_comparison.png'))
plt.close()
print("  Saved: metrics_comparison.png")

# --- Decision Tree feature importances ---
feat_imp_sorted = feat_imp.sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
feat_imp_sorted.plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Decision Tree — Feature Importances (Gini)')
ax.set_xlabel('Importance')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_importance_dt.png'))
plt.close()
print("  Saved: feature_importance_dt.png")

# --- GPC permutation importance ---
perm_result = permutation_importance(
    gpc_model, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE, scoring='roc_auc'
)
perm_imp = pd.Series(perm_result.importances_mean, index=SELECTED_FEATURES)
perm_err = pd.Series(perm_result.importances_std,  index=SELECTED_FEATURES)
sorted_idx = perm_imp.argsort()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(np.array(SELECTED_FEATURES)[sorted_idx],
        perm_imp.values[sorted_idx],
        xerr=perm_err.values[sorted_idx],
        color='#4C72B0', edgecolor='white', capsize=4)
ax.set_title('GPC — Permutation Feature Importances (ROC-AUC drop)')
ax.set_xlabel('Mean Decrease in ROC-AUC')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_importance_gpc.png'))
plt.close()
print("  Saved: feature_importance_gpc.png")

# --- Cross-validation score boxplot ---
fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot([gpc_cv_scores, dt_cv_scores], labels=['GPC', 'DTC'], patch_artist=True,
           boxprops=dict(facecolor='#4C72B0', alpha=0.6),
           medianprops=dict(color='black', linewidth=2))
ax.set_title('5-Fold Cross-Validation ROC-AUC')
ax.set_ylabel('ROC-AUC Score')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cv_scores.png'))
plt.close()
print("  Saved: cv_scores.png")

# --- Predicted probability histograms ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, probs, title, color in zip(
    axes,
    [gpc_prob, dt_prob],
    ['GPC Predicted Probabilities', 'DTC Predicted Probabilities'],
    ['#4C72B0', '#DD8452']
):
    for cls, ls, lbl in zip([0, 1], ['--', '-'],
                             ['Actual: No Epilepsy', 'Actual: Epilepsy']):
        ax.hist(probs[y_test == cls], bins=20, alpha=0.6, color=color,
                linestyle=ls, label=lbl, edgecolor='white')
    ax.axvline(0.5, color='black', linestyle=':', lw=1.5, label='Decision boundary')
    ax.set_title(title)
    ax.set_xlabel('Predicted Probability (Class 1)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'predicted_probabilities.png'))
plt.close()
print("  Saved: predicted_probabilities.png")


STEP 5 — GENERATING VISUALISATIONS
  Saved: confusion_matrices.png
  Saved: roc_curves.png
  Saved: metrics_comparison.png
  Saved: feature_importance_dt.png
  Saved: feature_importance_gpc.png
  Saved: cv_scores.png
  Saved: predicted_probabilities.png


In [22]:
print("\n" + "=" * 60)
print("FINAL MODEL COMPARISON SUMMARY")
print("=" * 60)

summary = pd.DataFrame({
    'Metric': ['Accuracy', 'ROC-AUC', 'Precision', 'Recall', 'F1-Score',
               'CV ROC-AUC (mean)', 'CV ROC-AUC (std)'],
    'GPC':    [f'{gpc_acc:.4f}', f'{gpc_auc:.4f}', f'{gpc_prec:.4f}',
               f'{gpc_rec:.4f}',  f'{gpc_f1:.4f}',
               f'{gpc_cv_scores.mean():.4f}', f'{gpc_cv_scores.std():.4f}'],
    'DTC':    [f'{dt_acc:.4f}',  f'{dt_auc:.4f}',  f'{dt_prec:.4f}',
               f'{dt_rec:.4f}',   f'{dt_f1:.4f}',
               f'{dt_cv_scores.mean():.4f}',  f'{dt_cv_scores.std():.4f}'],
})
print(summary.to_string(index=False))

winner = 'GPC' if gpc_auc > dt_auc else 'Decision Tree'
print(f"\nBest model by ROC-AUC: {winner}")

summary.to_csv(os.path.join(RESULTS_DIR, 'model_comparison.csv'), index=False)
print("  Saved: model_comparison.csv")
print("\n" + "=" * 60)
print("ALL RESULTS SAVED TO:", RESULTS_DIR)
print("=" * 60)


FINAL MODEL COMPARISON SUMMARY
           Metric    GPC    DTC
         Accuracy 0.6746 0.6360
          ROC-AUC 0.5497 0.5769
        Precision 0.0312 0.0352
           Recall 0.3636 0.4675
         F1-Score 0.0574 0.0655
CV ROC-AUC (mean) 0.5400 0.5290
 CV ROC-AUC (std) 0.0389 0.0450

Best model by ROC-AUC: Decision Tree
  Saved: model_comparison.csv

ALL RESULTS SAVED TO: c:\Users\shankar.ghimire\Downloads\data\results
